# Feature Engineering & Categorization for Naive Bayes

**Input:** `sleep_stage_sample_cleaned.csv` (500,000 rows)

**Pipeline:**
1. Select 6 independent features
2. 80/20 stratified train/test split
3. Domain-based binning (clinical thresholds)
4. Export 4 CSVs
5. Generate Naive Bayes frequency/probability tables from training data
6. Verification checks

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## 2. Load Dataset
Load the cleaned 500K-row dataset and inspect its shape and class distribution.

In [ ]:
df = pd.read_csv('sleep_stage_sample_cleaned.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df['sleep_stage'].value_counts().sort_index())

## 3. Feature Selection

We select **6 features** that are independent of each other (Naive Bayes assumption):

| Feature | Type | Why Selected |
|---------|------|--------------|
| `hr_mean` | Physiological | Core heart rate metric |
| `hr_sdnn_5` | Physiological | HR variability (dropped `hr_rmssd_5` due to r=0.95 correlation) |
| `hr_slope_3` | Physiological | HR trend direction (most independent, all corr ≤ 0.20) |
| `rr_mean` | Physiological | Core respiratory rate |
| `rr_slope_3` | Physiological | RR trend direction (corr ≤ 0.28 with others) |
| `minutes_since_start` | Temporal | Sleep architecture is time-dependent (corr ≤ 0.12) |

**Dropped:** `hr_rmssd_5` (redundant), `hr_rr_ratio` & `hr_rr_product` (derived, violate independence), `rr_sd_5` (moderate corr), `night_id`/`split`/`stage_name` (non-features)

In [ ]:
FEATURES = ['hr_mean', 'hr_sdnn_5', 'hr_slope_3', 'rr_mean', 'rr_slope_3', 'minutes_since_start']
TARGET = 'sleep_stage'

X = df[FEATURES]
y = df[TARGET]

print(f"Selected features: {FEATURES}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

## 4. Train/Test Split (80/20, Stratified)

We split **before** binning to prevent data leakage. Stratified to preserve class proportions in both sets.

- **Train:** 400,000 rows (80%) — used to build Naive Bayes tables
- **Test:** 100,000 rows (20%) — held out for future evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"\n--- Stratification Check (proportions should match) ---")
print(f"\nTrain distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print(f"\nTest distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(4))

## 5. Domain-Based Binning

We categorize continuous features using **clinical/domain thresholds**:

| Feature | Low/Decreasing/Early | Normal/Moderate/Stable/Mid | High/Increasing/Tachycardia/Late |
|---------|---------------------|--------------------------|--------------------------------|
| `hr_mean` | < 60 (Bradycardia) | 60–100 (Normal) | > 100 (Tachycardia) |
| `hr_sdnn_5` | < 1.0 (Low) | 1.0–3.0 (Moderate) | > 3.0 (High) |
| `hr_slope_3` | < −0.5 (Decreasing) | −0.5 to 0.5 (Stable) | > 0.5 (Increasing) |
| `rr_mean` | < 15 (Low) | 15–25 (Normal) | > 25 (High) |
| `rr_slope_3` | < −0.5 (Decreasing) | −0.5 to 0.5 (Stable) | > 0.5 (Increasing) |
| `minutes_since_start` | 0–180 (Early) | 180–360 (Mid) | > 360 (Late) |

Boundaries are fixed by domain knowledge — applied identically to both train and test.

In [ ]:
BIN_DEFINITIONS = {
    'hr_mean': {
        'bins': [-np.inf, 60, 100, np.inf],
        'labels': ['Bradycardia', 'Normal', 'Tachycardia']
    },
    'hr_sdnn_5': {
        'bins': [-np.inf, 1.0, 3.0, np.inf],
        'labels': ['Low', 'Moderate', 'High']
    },
    'hr_slope_3': {
        'bins': [-np.inf, -0.5, 0.5, np.inf],
        'labels': ['Decreasing', 'Stable', 'Increasing']
    },
    'rr_mean': {
        'bins': [-np.inf, 15, 25, np.inf],
        'labels': ['Low', 'Normal', 'High']
    },
    'rr_slope_3': {
        'bins': [-np.inf, -0.5, 0.5, np.inf],
        'labels': ['Decreasing', 'Stable', 'Increasing']
    },
    'minutes_since_start': {
        'bins': [-np.inf, 180, 360, np.inf],
        'labels': ['Early', 'Mid', 'Late']
    }
}


def apply_binning(dataframe, bin_definitions):
    """Apply domain-based binning to a dataframe of continuous features."""
    df_binned = dataframe.copy()
    for col, config in bin_definitions.items():
        df_binned[col] = pd.cut(
            dataframe[col],
            bins=config['bins'],
            labels=config['labels']
        )
    return df_binned


X_train_cat = apply_binning(X_train, BIN_DEFINITIONS)
X_test_cat = apply_binning(X_test, BIN_DEFINITIONS)

print("X_train categorized (first 10 rows):")
print(X_train_cat.head(10))

## 6. Verification

Confirm no NaN values were introduced by binning, and inspect bin distributions in training data.

In [ ]:
# Check for NaN values after binning
print("=== NaN Check ===")
print(f"X_train NaNs: {X_train_cat.isna().sum().sum()}")
print(f"X_test NaNs:  {X_test_cat.isna().sum().sum()}")

# Bin distributions per feature (training data)
print("\n=== Bin Distributions (Training Data) ===")
for col in FEATURES:
    print(f"\n--- {col} ---")
    counts = X_train_cat[col].value_counts().sort_index()
    print(counts)
    print(f"Total: {counts.sum()}")

## 7. Export CSVs

Save 4 CSV files:
- `x_train_categorized.csv` — 6 binned features (training)
- `y_train.csv` — sleep stage labels (training)
- `x_test_categorized.csv` — 6 binned features (test)
- `y_test.csv` — sleep stage labels (test)

In [ ]:
X_train_cat.to_csv('x_train_categorized.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
X_test_cat.to_csv('x_test_categorized.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("Exported 4 CSV files:")
print(f"  x_train_categorized.csv  ({X_train_cat.shape[0]} rows x {X_train_cat.shape[1]} cols)")
print(f"  y_train.csv              ({y_train.shape[0]} rows)")
print(f"  x_test_categorized.csv   ({X_test_cat.shape[0]} rows x {X_test_cat.shape[1]} cols)")
print(f"  y_test.csv               ({y_test.shape[0]} rows)")

## 8. Naive Bayes Frequency & Probability Tables

For each feature, we build a table showing:
- **Counts:** How many samples of each category appear per sleep stage
- **P(category | class):** The conditional probability = count / total for that class

These tables are built from **training data only**.

In [ ]:
STAGE_NAMES = {0: 'Wake', 1: 'N1', 2: 'N2', 3: 'N3', 5: 'REM'}
CLASS_ORDER = ['Wake', 'N1', 'N2', 'N3', 'REM']


def build_naive_bayes_table(feature_col, x_data, y_data):
    """Build a frequency + probability table for one feature."""
    combined = pd.DataFrame({
        'feature': x_data[feature_col],
        'class': y_data.map(STAGE_NAMES)
    })

    # Frequency crosstab
    freq = pd.crosstab(combined['feature'], combined['class'])
    freq = freq.reindex(columns=CLASS_ORDER)

    # Add totals row
    freq.loc['Total'] = freq.sum()

    # Probability: P(category | class) = count / class_total
    class_totals = freq.loc['Total']
    prob = freq.iloc[:-1].div(class_totals)
    prob = prob.round(4)
    prob.columns = [f'P({c})' for c in CLASS_ORDER]

    # Combine into one display table
    result = pd.concat([freq, prob], axis=1)
    return result

In [ ]:
# Generate and display tables for all 6 features
for feature in FEATURES:
    print(f"\n{'=' * 90}")
    print(f"  {feature}")
    print(f"{'=' * 90}")
    table = build_naive_bayes_table(feature, X_train_cat, y_train)
    print(table.to_string())
    print()

## 9. Class Prior Probabilities

P(class) — the prior probability of each sleep stage in the training data.
These are needed alongside the conditional probability tables for Naive Bayes classification.

In [ ]:
print("Class Prior Probabilities P(class) -- Training Data")
print("=" * 50)
priors = y_train.value_counts(normalize=True).sort_index()
priors.index = priors.index.map(STAGE_NAMES)
for stage, prob in priors.items():
    print(f"  P({stage:>4}) = {prob:.4f}")
print(f"\n  Total  = {priors.sum():.4f}")

## 10. Verification Summary

Automated checks to confirm the pipeline ran correctly:
1. Total rows (train + test) = 500,000
2. Stratification preserved across train/test
3. No empty bin-class combinations
4. P(category | class) sums to 1.0 per class

In [ ]:
print('=' * 70)
print('  VERIFICATION SUMMARY')
print('=' * 70)

# Check 1: Total rows
total_rows = X_train_cat.shape[0] + X_test_cat.shape[0]
status1 = 'PASS' if total_rows == 500000 else 'FAIL'
print(f'\n[{status1}] Total rows (train + test): {total_rows} (expected 500,000)')

# Check 2: Stratification preserved
train_props = y_train.value_counts(normalize=True).sort_index()
test_props = y_test.value_counts(normalize=True).sort_index()
max_diff = (train_props - test_props).abs().max()
status2 = 'PASS' if max_diff < 0.001 else 'FAIL'
print(f'[{status2}] Max stratification difference: {max_diff:.6f} (should be ~0)')

# Check 3: No empty bins per class
total_empty = 0
print(f'\nEmpty bin-class combinations per feature:')
for feature in FEATURES:
    combined = pd.DataFrame({
        'feature': X_train_cat[feature],
        'class': y_train.map(STAGE_NAMES)
    })
    ct = pd.crosstab(combined['feature'], combined['class'])
    empty = (ct == 0).sum().sum()
    total_empty += empty
    status = 'PASS' if empty == 0 else 'WARNING'
    print(f'  [{status}] {feature}: {empty} empty')
status3 = 'PASS' if total_empty == 0 else 'WARNING'
print(f'[{status3}] Total empty bin-class combos: {total_empty}')

# Check 4: P(category|class) sums to 1.0
print(f'\nP(category|class) sum-to-1 check:')
all_pass = True
for feature in FEATURES:
    table = build_naive_bayes_table(feature, X_train_cat, y_train)
    prob_cols = [c for c in table.columns if c.startswith('P(')]
    prob_sums = table.iloc[:-1][prob_cols].sum()
    ok = all(abs(s - 1.0) < 0.001 for s in prob_sums)
    if not ok:
        all_pass = False
    status = 'PASS' if ok else 'FAIL'
    print(f'  [{status}] {feature}')
status4 = 'PASS' if all_pass else 'FAIL'
print(f'[{status4}] All probability columns sum to 1.0')

print(f'\n{"=" * 70}')
all_ok = all(s == 'PASS' for s in [status1, status2, status3, status4])
print(f'  ALL CHECKS: {"PASSED" if all_ok else "ISSUES FOUND"}')
print(f'{"=" * 70}')